In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch10-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "activity-cliffs.py"

# 比對 BACE-1 活性與最近訓練鄰居

用同一標的的實測 pIC₅₀，分五折保留骨架，再逐一找出測試分子的最近訓練鄰居。分組檢查已保留的實測值，只用於事後分析。 首次執行需連網下載 MoleculeNet 資料，可以用環境變數 `DEEPCHEM_DATA_DIR` 指定快取目錄。

下一格寫出可修改的程式，末格執行並顯示圖片。做練習時，把自己的程式接在繪圖程式底部，重新執行這兩格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
cache = Path(os.environ.setdefault("DEEPCHEM_DATA_DIR", str(Path(tempfile.gettempdir()) / "cheminfo-molnet")))
cache.mkdir(parents=True, exist_ok=True)
import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestRegressor

SEED = 42
figures = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figures.mkdir(parents=True, exist_ok=True)
from rdkit.Chem import Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupKFold

tasks, (dataset,), _ = dc.molnet.load_bace_regression(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None, transformers=[],
    data_dir=str(cache), save_dir=str(cache), reload=True)
table = pd.DataFrame({"smiles": dataset.X, "pIC50": dataset.y[:, tasks.index("pIC50")]})
labels = table.groupby("smiles", sort=True)["pIC50"].median()
mols = [Chem.MolFromSmiles(s) for s in labels.index]
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024, includeChirality=True)
fps = [gen.GetFingerprint(m) for m in mols]
X = np.stack([gen.GetFingerprintAsNumPy(m) for m in mols])
y = labels.to_numpy()
scaffolds = [MurckoScaffold.MurckoScaffoldSmiles(mol=m) for m in mols]
pred = np.empty(len(y))
nearest = np.empty(len(y), dtype=int)
similarity = np.empty(len(y))
for train, test in GroupKFold(5).split(X, y, scaffolds):
    model = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=1)
    model.fit(X[train], y[train])
    pred[test] = model.predict(X[test])
    train_fps = [fps[j] for j in train]
    for i in test:
        values = DataStructs.BulkTanimotoSimilarity(fps[i], train_fps)
        best = int(np.argmax(values))
        nearest[i], similarity[i] = train[best], values[best]

delta = np.abs(y - y[nearest])
error = np.abs(y - pred)
near = similarity >= 0.60
agree = near & (delta < 1.0)
cliff = near & (delta >= 1.0)
print(f"rows={len(table)}, structures={len(y)}, scaffolds={len(set(scaffolds))}")
for label, mask in [("all", np.ones(len(y), bool)), ("agree", agree), ("cliff", cliff)]:
    print(f"{label}: n={mask.sum()}, MAE={error[mask].mean():.3f}")
# 標出相似度最高的懸崖案例；平手時採標準 SMILES 字典序。
i = np.flatnonzero(cliff)[np.argmax(similarity[cliff])]
j = nearest[i]
print(f"example: similarity={similarity[i]:.3f}, measured={y[i]:.3f}, nearest={y[j]:.3f}, predicted={pred[i]:.3f}")
print("test:", labels.index[i], "\ntraining:", labels.index[j])
fig, (structures, ax) = plt.subplots(2, 1, figsize=(7.4, 7.5), gridspec_kw={"height_ratios": [2, 3]})
pair = Draw.MolsToGridImage([mols[i], mols[j]], molsPerRow=2, subImgSize=(460, 260),
    legends=[f"Held-out: pIC50 {y[i]:.2f}", f"Nearest training: pIC50 {y[j]:.2f}"], returnPNG=False)
structures.imshow(pair)
structures.axis("off")
for mask, color, label in [(agree, "#277a81", "Difference < 1"), (cliff, "#af4c30", "Difference ≥ 1")]:
    ax.scatter(delta[mask], error[mask], s=15, alpha=.55, color=color, label=f"{label} (n={mask.sum()})")
ax.axvline(1, ls="--", lw=1, color="#38464e")
ax.scatter(delta[i], error[i], s=100, facecolors="none", edgecolors="black", label="Example in text")
ax.set(xlabel="Absolute pIC50 difference to nearest training molecule", ylabel="Absolute out-of-fold error (pIC50)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(figures / "10-07-activity-cliffs.png", dpi=150)

In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "10-07-activity-cliffs.png")))